# Skill Selection

**Last updated:** 2026-04-09 12:00 PT

**Track:** Learning | **Sub-Ability:** Procedural learning

Can the model learn to select the correct tool/skill from a growing registry as context gets noisier?
Pre/post learning framework: study practice examples, then pick the right tool.

**Difficulty grid:** num_tools × similarity × 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()


def extract_tool_name(response: str, valid_names: list) -> str:
    """Extract a tool name from model response using fuzzy matching."""
    text = strip_thinking(response)
    # Remove markdown formatting but NOT underscores (they're in tool names!)
    text = re.sub(r'[`*]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]

    # Try exact match first (case-insensitive)
    for line in reversed(lines):
        clean = line.strip('"\'').strip().lower()
        for name in valid_names:
            if clean == name.lower():
                return name

    # Try substring match on short lines (likely the answer line)
    for line in reversed(lines):
        if len(line) > 60:
            continue
        clean = line.strip('"\'').strip().lower()
        for name in valid_names:
            if name.lower() in clean:
                return name

    # Try substring match on any line
    for line in reversed(lines):
        clean = line.lower()
        for name in valid_names:
            if name.lower() in clean:
                return name

    return ''

In [ ]:
# Constants used in results analysis
NUM_TOOLS = [5, 15, 30]
SIMILARITIES = ['distinct', 'confusable', 'adversarial']
SEEDS = 3

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
# Try multiple possible paths (varies by how dataset was attached)
candidates = glob.glob('/kaggle/input/**/skill_selection_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/skill-selection-benchmark/skill_selection_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')
print(dataset[['task_id', 'difficulty_label', 'expected', 'test_domain', 'item_desc']].to_string(index=False))

In [ ]:
PRE_P = """You are using the Nexara Platform. You must select the correct tool for the task below.

{material}

--- TASK ---
{test_input}
--- END TASK ---

Based on the tool registry above, which tool should be used for this task?
Reply with ONLY the tool name (e.g. flux_bridge). Nothing else."""

STUDY_P = """You are learning to use the Nexara Platform tool registry.

{study_material}

Analyze these practice examples carefully:
1. For each example, explain WHY the correct tool was chosen.
2. Identify any patterns in how tool names, descriptions, and capabilities relate.
3. Note any cases where a tool's name or description is misleading — capabilities are what matter.
4. Write a quick-reference guide: for each tool, summarize what it ACTUALLY does in one line.

Write thorough notes you can refer to later when selecting tools."""

POST_P = """You previously studied the Nexara Platform tool registry and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Now select the correct tool for this task:

{material}

--- TASK ---
{test_input}
--- END TASK ---

Based on your study notes and the registry, which tool should be used?
Reply with ONLY the tool name (e.g. flux_bridge). Nothing else."""

## Evaluation

In [ ]:
@kbench.task(name='skill_selection_v2',
             description='Pre/post learning tool selection from growing registries')
def skill_selection(llm) -> float:
    model_name = str(llm)
    correct = 0
    total = len(dataset)
    all_results = []

    for _, row in dataset.iterrows():
        material = row['material']
        study_material = row['study_material']
        test_input = row['test_input']
        expected = row['expected']
        num_tools = int(row['num_tools'])
        similarity = row['similarity']
        difficulty_label = row['difficulty_label']
        seed = int(row['seed'])
        item_desc = row['item_desc']
        test_domain = row['test_domain']
        names_list = json.loads(row['valid_names'])
        ts = time.strftime('%Y-%m-%d %H:%M:%S')
        tid = f'{num_tools}tools_{similarity}_{seed}'

        # --- PRE: answer without study ---
        t0 = time.time()
        pre_raw = llm.prompt(PRE_P.format(material=material, test_input=test_input))
        pre_time = time.time() - t0
        pre_extracted = extract_tool_name(pre_raw, names_list)
        pre_correct = pre_extracted == expected

        # --- STUDY: analyze practice examples ---
        t0 = time.time()
        study_raw = llm.prompt(STUDY_P.format(study_material=study_material))
        study_time = time.time() - t0
        notes = strip_thinking(study_raw)

        # --- POST: answer with study notes ---
        t0 = time.time()
        post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_input=test_input))
        post_time = time.time() - t0
        post_extracted = extract_tool_name(post_raw, names_list)
        post_correct = post_extracted == expected

        if post_correct:
            correct += 1

        all_results.append({
            'timestamp': ts, 'model': model_name, 'task_id': tid,
            'num_tools': int(num_tools), 'similarity': similarity, 'difficulty_label': difficulty_label,
            'seed': int(seed), 'item_desc': item_desc, 'test_domain': test_domain,
            'test_input': test_input, 'expected': expected, 'valid_names': row['valid_names'],
            'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
            'study_notes': notes,
            'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
            'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time,
        })

        b,l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
        print(f'[{model_name:40s}] [{difficulty_label:25s}] exp="{expected}" pre="{pre_extracted}"({b}) post="{post_extracted}"({l})')
        # Per-item assertion for diagnostics
        kbench.assertions.assert_equal(post_extracted, expected)

    score = correct / total
    print(f'\nScore: {correct}/{total} = {score:.4f}')
    return score

try:
    runs = skill_selection.evaluate(llm=[kbench.llm],
                                   n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc
room = 1.0 - pre_acc
eff = gain / room if room > 0 else 0.0

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print(f'Learning efficiency:    {eff:.1%}')
print()

print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:25s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Test domain: {row["test_domain"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Skill Selection: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('skill_selection_results.png', dpi=150, bbox_inches='tight')
plt.show()